# HISTORA Oral-Systemic — J-lens input diagnostic

Uses the **Jacobian lens** on a small open-weights proxy (Qwen) to measure whether a
candidate input format makes the periodontal <-> cardiovascular *mediating concepts*
representable in the model's internal workspace band.

**Self-contained:** this notebook clones `jacobian-lens` and writes its own `src/`
modules with `%%writefile`. No private GitHub, no Drive, no manual uploads.
Run in Colab with a GPU. Non-diagnostic methodology work on synthetic data.


## 1. Setup (GPU runtime)

In [ ]:
# Clone the reference lens implementation and install it.
!git clone -q https://github.com/anthropics/jacobian-lens
!pip install -q -e jacobian-lens

## 2. Write this project's `src/` modules into the Colab working dir

These cells recreate `bridge_concepts.py`, `record_formats.py`, and `harness.py`
locally, so the imports below work with nothing to fetch. Edit them here (or upstream
in the repo and regenerate) as the concept lists and formats evolve.

In [ ]:
%%writefile bridge_concepts.py
"""Target bridge concepts for the oral-systemic (periodontal <-> cardiovascular) task.

These are the mediating concepts that must appear in the workspace band of the
J-lens readout for an input format to count as "good": if the model represents
them internally, it is *relating* oral and systemic data, not merely listing it.

Each concept maps to several candidate single-token surface forms; the harness
keeps only the ones that tokenize to a single token under the target model.

MEDIATORS are the unspoken, cross-domain links (inflammation, atherosclerosis,
endothelial dysfunction, bacteremia). SHARED are common risk factors that appear
in the input itself (diabetes, smoking); they matter less as evidence of
relational reasoning because a model can surface them by mere copying.
"""

from __future__ import annotations

BRIDGE_CONCEPTS: dict[str, list[str]] = {
    # --- mediators (unspoken cross-domain links) ---
    "systemic_inflammation": ["inflammation", "inflammatory"],
    "c_reactive_protein": ["CRP", "protein"],  # "hs-CRP" is usually multi-token
    "cytokines": ["cytokine", "interleukin", "IL"],
    "atherosclerosis": ["atherosclerosis", "plaque", "atheroma"],
    "endothelial_dysfunction": ["endothelial", "endothelium", "vascular"],
    "bacteremia": ["bacteremia", "bacteria", "microbial"],
    "oxidative_stress": ["oxidative", "oxidation"],
    "cardiovascular_risk": ["cardiovascular", "coronary", "arterial", "cardiac"],
    # --- shared risk factors (present in the input; weaker evidence) ---
    "shared_diabetes": ["diabetes", "glycemic", "insulin"],
    "shared_smoking": ["smoking", "tobacco", "nicotine"],
}

# Concepts that prove relational reasoning; the optimizer targets these first.
MEDIATOR_KEYS = [
    "systemic_inflammation",
    "c_reactive_protein",
    "cytokines",
    "atherosclerosis",
    "endothelial_dysfunction",
    "bacteremia",
    "oxidative_stress",
    "cardiovascular_risk",
]


In [ ]:
%%writefile record_formats.py
"""One synthetic combined medical + periodontal record, rendered in three
candidate input formats.

The three formats isolate the levers we optimize:
  A  abbreviated table, dental-first, no glossing of clinical terms
  B  named sections + glossed terms, medical-first
  C  narrative prose with an explicit mechanistic knowledge-base bridge

Hypothesis: C >= B >> A in the workspace rank of the mediator concepts.

This is SYNTHETIC data for methodology development. The task is non-diagnostic:
outputs are research hypotheses and data-completeness flags, never diagnoses.
"""

from __future__ import annotations

RECORD = {
    "demographics": {"age": 58, "sex": "M", "bmi": 29.4},
    "shared_risk": {
        "smoking_pack_years": 20,
        "smoking_active": True,
        "type2_diabetes": True,
        "hba1c": 8.1,
        "hypertension": True,
        "blood_pressure": "140/90",
    },
    "medical_cv": {
        "ldl": 155,
        "hdl": 38,
        "triglycerides": 210,
        "hs_crp": None,  # deliberately missing -> should trigger a completeness flag
        "prior_cv_event": False,
        "family_history_mi": True,
        "medications": ["enalapril", "metformin"],
        "on_statin": False,
    },
    "periodontal": {
        "mean_ppd_mm": 5.2,
        "ppd_18m_ago_mm": 4.1,
        "cal_mm": 4.8,
        "bop_pct": 62,
        "radiographic_bone_loss": "moderate-to-severe, generalized",
        "diagnosis": "periodontitis stage III grade B",
        "regular_maintenance": False,
    },
}

TARGET_ANSWER_HINT = "elevated oral-systemic risk driven by the inflammatory axis"


def format_a_abbrev_table(r: dict) -> str:
    """Abbreviated, dental-first, no term glossing."""
    return (
        "PERIO: PPD 5.2(^4.1) CAL 4.8 BOP 62% BoneLoss mod-sev; dx perio III/B\n"
        "MED: HbA1c 8.1 LDL 155 HDL 38 TG 210 HTN 140/90 smoker 20py CRP ?\n"
        "Q: oral-systemic risk?"
    )


def format_b_sections_glossed(r: dict) -> str:
    """Named sections + glossed terms, medical-first."""
    return (
        "## SHARED RISK FACTORS\n"
        "smoking: 20 pack-years (active) | type 2 diabetes: yes, HbA1c 8.1% | "
        "hypertension 140/90\n"
        "## CARDIOVASCULAR PROFILE\n"
        "LDL 155, HDL 38, TG 210 | hs-CRP: MISSING | family history of MI: yes | "
        "statin: no\n"
        "## PERIODONTAL PROFILE (longitudinal)\n"
        "mean PPD 5.2 mm (up from 4.1 mm 18 months ago) | CAL 4.8 mm | "
        "BOP 62% (bleeding on probing = marker of gingival inflammation) | "
        "bone loss moderate-to-severe | periodontitis stage III grade B\n"
        "## QUESTION\nWhat is the oral-systemic risk profile and which relational axes apply?"
    )


def format_c_narrative_mechanism(r: dict) -> str:
    """Narrative prose with an explicit mechanistic KB bridge."""
    return (
        "Context: periodontitis can raise systemic inflammatory burden, which is "
        "associated with cardiovascular risk through markers such as C-reactive protein.\n"
        "A 58-year-old man, smoker (20 pack-years), type 2 diabetic with HbA1c 8.1%, "
        "hypertensive, LDL 155 / HDL 38, with periodontitis stage III grade B, BOP 62%, "
        "and progressive moderate-to-severe bone loss. No CRP on record and not on a statin.\n"
        "What oral-systemic risk profile and relational axes does this suggest?"
    )


FORMATS = {
    "A_abbrev_table": format_a_abbrev_table,
    "B_sections_glossed": format_b_sections_glossed,
    "C_narrative_mechanism": format_c_narrative_mechanism,
}

# Key patient values used by the capacity probe (single-token-friendly surfaces).
DATA_ITEMS = ["58", "20", "8.1", "155", "38", "210", "140", "62", "5.2", "4.8"]


In [ ]:
%%writefile harness.py
"""J-lens diagnostic harness for the oral-systemic input-format optimization loop.

Reads out, for each candidate input format, the minimum vocabulary rank of every
target bridge concept within the model's workspace band. A low rank means the
concept is represented internally and is available for verbal report; a high or
absent rank means the format fails to make that relation representable.

Requires a GPU (run in Colab). Depends on the `jlens` package from the
jacobian-lens repo: `pip install -e jacobian-lens`.
"""

from __future__ import annotations

import math
from typing import Any

from bridge_concepts import BRIDGE_CONCEPTS, MEDIATOR_KEYS
from record_formats import DATA_ITEMS


def single_token_id(tokenizer: Any, surface: str) -> int | None:
    """Return the token id if `surface` (with or without a leading space) encodes
    to exactly one token, else None. Multi-token surfaces are not directly
    measurable by the lens (it reads one vocabulary token per direction)."""
    for candidate in (" " + surface, surface):
        ids = tokenizer.encode(candidate, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None


def workspace_band(model: Any, lens: Any, lo: float = 0.33, hi: float = 0.66) -> list[int]:
    """Fitted layers whose depth fraction lies in the mid-network workspace band.
    Calibrate [lo, hi] once per model with `sweep_layers`."""
    return [l for l in lens.source_layers if lo <= l / model.n_layers <= hi]


def concept_ranks(
    model: Any,
    lens: Any,
    tokenizer: Any,
    prompt: str,
    band: list[int],
    tail_tokens: int = 40,
) -> dict[str, dict]:
    """Minimum lens rank of each bridge concept over the workspace band and the
    last `tail_tokens` positions (the question / answer span)."""
    lens_logits, _, input_ids = lens.apply(model, prompt, layers=band, positions=None)
    seq = input_ids.shape[-1]
    tail = list(range(max(0, seq - tail_tokens), seq))
    out: dict[str, dict] = {}
    for name, surfaces in BRIDGE_CONCEPTS.items():
        best, best_surface = math.inf, None
        for s in surfaces:
            tid = single_token_id(tokenizer, s)
            if tid is None:
                continue
            for l in band:
                sub = lens_logits[l][tail]  # [n_tail, vocab]
                ranks = (sub > sub[:, tid : tid + 1]).sum(dim=1)  # rank per position
                r = int(ranks.min())
                if r < best:
                    best, best_surface = r, s
        out[name] = {
            "min_rank": best,
            "surface": best_surface,
            "hit_at_1": best == 0,
            "hit_at_10": best < 10,
        }
    return out


def capacity(
    model: Any,
    lens: Any,
    tokenizer: Any,
    prompt: str,
    band: list[int],
    k: int = 25,
    tail_tokens: int = 60,
) -> tuple[int, int]:
    """How many key patient data items remain reachable (band-min rank < k)."""
    lens_logits, _, ids = lens.apply(model, prompt, layers=band, positions=None)
    seq = ids.shape[-1]
    tail = list(range(max(0, seq - tail_tokens), seq))
    reachable = 0
    measured = 0
    for item in DATA_ITEMS:
        tid = single_token_id(tokenizer, item)
        if tid is None:
            continue
        measured += 1
        best = min(
            int((lens_logits[l][tail] > lens_logits[l][tail][:, tid : tid + 1]).sum(1).min())
            for l in band
        )
        reachable += best < k
    return reachable, measured


def sweep_layers(
    model: Any,
    lens: Any,
    tokenizer: Any,
    prompt: str,
    surface: str,
    tail_tokens: int = 40,
) -> dict[int, int]:
    """Min rank of one surface at every fitted layer, to calibrate the band."""
    tid = single_token_id(tokenizer, surface)
    if tid is None:
        raise ValueError(f"{surface!r} is not a single token for this model")
    lens_logits, _, ids = lens.apply(model, prompt, layers=lens.source_layers, positions=None)
    seq = ids.shape[-1]
    tail = list(range(max(0, seq - tail_tokens), seq))
    return {
        l: int((lens_logits[l][tail] > lens_logits[l][tail][:, tid : tid + 1]).sum(1).min())
        for l in lens.source_layers
    }


def summarize(name: str, ranks: dict[str, dict]) -> str:
    """One-line summary line plus mediator ranks for a format."""
    hits = sum(v["hit_at_10"] for v in ranks.values())
    mediator_ranks = sorted(ranks[c]["min_rank"] for c in MEDIATOR_KEYS)
    return f"{name}: hit@10={hits}/{len(ranks)}  mediator_ranks={mediator_ranks}"


## 3. Load the proxy model and a pre-fitted lens

Pre-fitted lenses are published on the Hub, so no fitting is needed. Use the 27B if the
GPU allows; the 4B is a weaker proxy and may under-represent clinical relations.

In [ ]:
import torch, transformers, jlens
from jlens import JacobianLens

MODEL_NAME = 'Qwen/Qwen3.6-27B'   # fallback: 'Qwen/Qwen3.5-4B'
LENS_REPO, LENS_REV = 'neuronpedia/jacobian-lens', 'qwen-n1000'
LENS_FILE = {
    'Qwen/Qwen3.6-27B': 'qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt',
    'Qwen/Qwen3.5-4B':  'qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt',
}[MODEL_NAME]

hf = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).cuda()
tok = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf, tok)
lens = JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE, revision=LENS_REV)
model, lens

## 4. Load targets, records, and metrics

In [ ]:
from bridge_concepts import BRIDGE_CONCEPTS, MEDIATOR_KEYS
from record_formats import RECORD, FORMATS, DATA_ITEMS
from harness import concept_ranks, capacity, workspace_band, sweep_layers, summarize

band = workspace_band(model, lens)
print('fitted layers:', lens.source_layers)
print('workspace band:', band)

## 5. Calibrate the band (optional)

Confirm where an abstract mediator actually surfaces before trusting the [0.33, 0.66] prior.

In [ ]:
ranks_by_layer = sweep_layers(model, lens, tok, FORMATS['C_narrative_mechanism'](RECORD), 'inflammation')
for l in sorted(ranks_by_layer):
    bar = '#' * max(0, 30 - min(30, ranks_by_layer[l]))
    print(f"L{l:>3}  rank={ranks_by_layer[l]:<6} {bar}")

## 6. Score the candidate formats

Lower rank = the concept is represented internally. Compare mediator ranks across A/B/C.

In [ ]:
results = {}
for fname, fn in FORMATS.items():
    r = concept_ranks(model, lens, tok, fn(RECORD), band)
    results[fname] = r
    print('\n=== ' + summarize(fname, r) + ' ===')
    for c, v in r.items():
        tag = 'HIT' if v['hit_at_10'] else ''
        star = '*' if c in MEDIATOR_KEYS else ' '
        print(f"  {star} {c:26s} rank={v['min_rank']:<6} via {str(v['surface']):>14}  {tag}")

## 7. Capacity check

How many key patient data items stay reachable at once. If far fewer than the record
size, the format must chunk / summarize / hierarchize.

In [ ]:
for fname, fn in FORMATS.items():
    reachable, measured = capacity(model, lens, tok, fn(RECORD), band)
    print(f'{fname}: {reachable}/{measured} data items reachable (rank < 25)')

## 8. Export results (paste into docs/PLAN.md progress log)

In [ ]:
import json
compact = {f: {c: v['min_rank'] for c, v in r.items()} for f, r in results.items()}
print(json.dumps(compact, indent=2))

## 9. Feed results to the Claude controller

Send `results[fname]` (the per-concept ranks) plus the proxy's actual answer and the
candidate format to Claude using `prompts/controller.md`. Apply its format edits, re-run
cells 6-7, and iterate until every mediator reaches hit@10. Then hand the converged
format to the Claude evaluator (`prompts/evaluator.md`) for the final structured output,
and validate with task accuracy measured on Claude — not on the proxy's ranks.

### Alternative to self-contained mode: Google Drive
If you prefer to keep editing the modules in your repo folder on Drive instead of the
`%%writefile` cells, replace section 2 with:
```python
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/dental-analysis/src')
```
